# Retrieval

LangChain에서 Retrieval은 외부 데이터에서 관련 정보를 찾아 프롬프트에 포함시켜(Context) LLM에 전달하는 역할을 한다. 주요 구성 요소는 다음과 같다.

- **Document Loader**: 다양한 원본 데이터를 LangChain 표준 문서 객체로 변환한다.
- **Text Splitter**: 긴 문서를 작은 청크로 분할해 검색 효율을 높인다.
- **Embedding Model**: 텍스트를 의미 기반 벡터로 변환한다.
- **Vector Store**: 임베딩된 벡터를 저장하고 유사도 기반 검색을 지원한다.
- **Retriever**: 쿼리에 대해 관련 문서를 찾아주는 표준 인터페이스를 제공한다.
- **Retrieval Chain**: 검색된 문서를 LLM에 전달해 답변을 생성하는 체인 구조를 제공한다.

이렇게 각 모듈이 결합되어, 외부 데이터 기반의 효과적인 검색 및 답변 생성이 가능하다.

**환각 Hallucination:**

LLM이 실제 근거 없이 그럴듯해 보이는 정보를 생성하는 현상이다.

**주요 원인**
1. **학습 데이터 한계**
   * 모델이 학습한 데이터에 해당 정보가 없거나 부족할 때 발생한다.
2. **확률적 생성 과정**
   * 토큰 예측 시 언어적 일관성을 우선하다 보니, 사실 여부가 검증되지 않은 내용을 생성한다.
3. **프롬프트 모호성**
   * 지시가 불명확하거나 맥락이 부족하면 모델이 관련 없는 정보를 보충·왜곡한다.

**대표 사례**
* 존재하지 않는 논문·저자명을 인용함.
* 역사적·과학적 사실을 잘못 기술함.
* 실행 불가능하거나 비효율적인 코드 제안.


**완화 방안**

1. **지식 기반 검색 결합**
   * Retrieval-Augmented Generation(RAG) 방식으로 외부 문서·데이터베이스에서 실시간 근거를 가져와 보강한다.
2. **프롬프트 구체화**
   * “출처를 함께 제시해 달라” 등 명시적 요청을 통해 근거 표기를 유도한다.
3. **후처리 검증**
   * 생성 결과를 룰 기반 검증 또는 전문가 리뷰를 통해 교차 확인한다.
4. **모델 파인튜닝 및 앙상블**
   * 도메인 특화 데이터로 추가 학습하거나, 룰 기반 시스템과 결합하여 정확도를 높인다.

In [ ]:
%pip install langchain langchain-community langchain-openai langchain-huggingface wikipedia pypdf tavily-python tiktoken faiss-cpu sentence-transformers -Uqqq

## Document

Document는 LangChain 프레임워크에서 다양한 데이터 소스(예: 텍스트 파일, PDF, 웹페이지 등)로부터 불러온 정보를 표준화된 객체로 표현하는 핵심 데이터 구조이다. 이 객체는 언어 모델(LLM)이 외부 데이터를 이해하고 처리할 수 있도록 도와준다.

**Document 객체의 구조**
1. page_content: 문서의 실제 내용을 담고 있는 문자열(str)이다. 예를 들어, 텍스트 파일의 본문이나 PDF의 텍스트 등이 여기에 저장된다.
2. metadata: 문서에 대한 부가 정보를 담는 딕셔너리(dict) 형태의 속성이다. 예를 들어, 파일 경로, 페이지 번호, 작성자, 데이터 출처 등 다양한 메타데이터를 저장할 수 있다.


**Document의 역할과 활용**
1. 표준화된 데이터 구조: 다양한 포맷의 데이터를 일관된 방식으로 표현하여, LLM이 손쉽게 접근하고 활용할 수 있도록 한다.
2. 문서 처리의 기본 단위: LangChain의 문서 로더(Document Loader)는 파일, 웹, 데이터베이스 등 여러 소스에서 데이터를 읽어와 Document 객체로 변환한다.
3. 청크 단위 분할: 대용량 문서는 작은 단위(청크)로 쪼개어 각각의 Document로 저장하고, 검색 및 임베딩 처리에 활용한다.

In [ ]:
from langchain_core.documents import Document       # Langchain 표준 문서 단위 객체

doc = Document(
    page_content = '이것은 랭체인의 Document 객체입니다. 모든 데이터소스는 이 Document 객체로 변환됩니다.',
    metadata = {
        'source' : 'durlwjrl',          # 데이터 출처
        'url' : 'https://encore.com',   # 원문 url
        'timestamp': 202608250939       # 수집/ 생성 시간
    }
)


## Document Loader
https://reference.langchain.com/python/langchain_core/document_loaders/


Document Loader는 다양한 데이터 소스에서 데이터를 읽어와 Document 객체로 변환하는 역할을 한다. 예를 들어, PDFLoader, CSVLoader, TextLoader 등 다양한 종류가 존재하며, 각기 다른 파일 형식을 Document 객체로 표준화한다.

Document Loader는 데이터 소스별로 특화된 클래스를 제공하며, 문서를 로드한 후 LangChain에서 사용하는 표준 형식으로 변환해준다.

1. **다양한 데이터 소스 지원**  
   Document Loader는 파일 시스템, 클라우드 스토리지, 데이터베이스, 웹 등 다양한 데이터 소스에서 데이터를 로드할 수 있도록 설계되었다.
   
2. **표준화된 출력 형식**  
   로드된 문서는 LangChain에서 사용하는 `Document` 객체로 변환된다. `Document` 객체는 다음과 같은 필드를 포함한다:
   - `page_content`: 문서 본문 내용
   - `metadata`: 문서와 관련된 메타데이터 (예: 파일 이름, URL, 작성자 등)

3. **플러그인 기반 확장 가능**  
   사용자 정의 데이터 소스 로더를 쉽게 구현하고 LangChain에 통합할 수 있다.

**주요 Document Loader 예시**

| Loader 이름        | 설명                                                              |
|--------------------|-------------------------------------------------------------------|
| `PyPDFLoader`      | PDF 문서를 로드하며 텍스트를 추출해 Document 형식으로 변환한다.     |
| `TextLoader`       | 일반 텍스트 파일을 로드한다.                                      |
| `UnstructuredFileLoader` | 비구조적 데이터를 로드하여 구조화된 텍스트로 변환한다.           |
| `CSVLoader`        | CSV 파일에서 데이터를 로드하며 행(row)을 Document로 처리한다.      |
| `WebBaseLoader`    | 웹 페이지 데이터를 크롤링하여 Document로 로드한다.                |

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

url = 'https://n.news.naver.com/article/047/0002526580'

header = {
    # 브라우저 식별 : Windows 에서 chrome으로 접속한 것처럼 보이게 만드는 UA
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

loader = WebBaseLoader(url, header_template=header)     # 로더 객체 생성
docs = loader.load()            # 불러온 웹페이지 -> Document 리스트
docs

In [ ]:
print(len(docs))

doc = docs[0]
print(doc.metadata)
print(doc.metadata['title'])
print(doc.page_content)

In [ ]:
!gdown 1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO

In [ ]:
from langchain_community.document_loaders import PyPDFLoader    # PDF 를 읽어 Document 로 로드하는 로더

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()    # PDF를 페이지별 Document 리스트로 변환
print(len(docs))    # Document 수

In [ ]:
print(docs[2].metadata)             # 3 페이지의 메타데아터
print(docs[2].metadata['source'])   # 경로
print(docs[2].metadata['page'])     # 현재 페이지
print(docs[2].metadata['page_label'])   # 사람이 볼수있는 페이지 번호
print(docs[2].page_content)         # 본문

### TavilySearchAPIRetriever
https://www.tavily.com/

- `langchain_tavily.TavilySearch`: Agent tool사용버젼. json반환
- `langchain_community.retrievers.TavilySearchAPIRetriever`: 검색기(context확보용) Document객체반환

- 주요 기능
    - 웹 검색(query → 결과 리스트): 키워드로 웹을 검색해서 관련 페이지들을 찾아줌
    - 요약/스니펫 제공: 각 결과에 본문 요약이나 핵심 스니펫을 같이 줘서 LLM이 바로 쓰기 좋음
    - 컨텐츠 추출(include_raw_content 등 옵션): 결과 페이지의 내용을 일부/전체 텍스트로 가져오게 설정 가능
    - 필터링/튜닝 옵션: 검색 결과 개수, 도메인 포함/제외, 최신성(리센시) 같은 옵션으로 결과를 조절 가능
    - RAG 파이프라인에 바로 연결: “검색 → 문서(Document)화 → 벡터화/리랭킹 → 답변” 흐름에서 검색 단계로 많이 사용

In [ ]:
!pip install tavily-python -qqq

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

In [ ]:
from langchain_community.retrievers import TavilySearchAPIRetriever

tavily_retriever = TavilySearchAPIRetriever(k = 3)

docs = tavily_retriever.invoke('런닝')
docs

In [ ]:
for doc in docs:
    print(doc.page_content)

### Tavily 검색 결과로 Context에 넣고 답변하는 RAG chain


In [ ]:
from langchain_core.prompts import PromptTemplate          
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

tavily_retriever = TavilySearchAPIRetriever(k= 3)   # 검색결과 상위 3개
prompt = PromptTemplate.from_template('''
        사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.    
        Context : {context}
        Question : {question}
    ''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser()

def format_docs(docs: list[Document]) -> str:
    return '\n\n'.join(doc.page_content for doc in docs)

tavily_chain = tavily_retriever | format_docs

chain = (
    {'question': RunnablePassthrough(), 'context': tavily_chain} | prompt | llm | output_parser
)

chain.invoke('8월 말 독산역 인기 맛집?')

## Embedding Model
- openai
- setence-transformer(huggingface)

In [ ]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')
text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text)
print(len(emb_vec))
print(emb_vec[:3])

pd.Series(emb_vec, name= 'embedding')

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model= 'sentence-transformers/all-MiniLM-L6-V3')
text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text)
print(len(emb_vec))
print(emb_vec[:3])

pd.Series(emb_vec, name= 'embedding')

## Vector Store

벡터 데이터베이스란 쉽게 말해, **비정형 데이터(텍스트, 이미지, 오디오 등)를 숫자 벡터로 변환하여 저장하고, 이 벡터들 간의 유사성을 바탕으로 데이터를 검색**하는 데이터베이스를 말한다. 여기서 벡터는 데이터를 다차원 공간에서 표현한 수학적 객체이다.

- **벡터**: 데이터의 특징을 다차원으로 표현한 값.
  - 예: 단어 임베딩은 단어를 벡터로 변환하여 유사한 단어들이 가까이 위치.
- **벡터 데이터베이스 필요성**:
  - RDBMS는 구조화된 데이터(테이블 형태)에 적합.
  - AI/머신러닝의 발전으로 비정형 데이터를 처리할 필요 증가.
  - 벡터 데이터베이스는 **유사도 기반 검색**으로 고차원 데이터 처리에 유리.

**주요 특징:**
- 유사한 데이터를 빠르게 검색.
- AI 응용 분야(이미지 검색, 자연어 처리, 추천 시스템 등)에서 중요.

- **벡터 데이터베이스와 RDBMS의 주요 차이점**

| **특징**                | **RDBMS**                                                                 | **벡터 데이터베이스**                                                                                  |
|-------------------------|---------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------|
| **데이터 구조**          | 테이블 형식으로 데이터 저장, SQL을 사용하여 질의.                             | 다차원 벡터 형식으로 데이터 저장, 벡터 간 유사도 계산 방식 사용.                                        |
| **검색 방식**            | 키-값 쌍이나 고정 조건 기반 검색 (정확한 일치 검색).                          | 유사성 검색 수행, 벡터 간 거리(예: 코사인 유사도, 유클리드 거리)를 기준으로 유사한 데이터를 반환.         |
| **비정형 데이터 처리**   | 텍스트, 숫자 등 구조화된 데이터 처리에 적합.                                 | 이미지, 오디오, 영상 등 비정형 데이터를 벡터로 변환해 처리 가능.                                       |
| **응용 분야**            | 전통적인 CRUD 작업, 금융 데이터, 고객 데이터 관리 등.                       | AI 기반 추천 시스템, 이미지 검색, 자연어 처리, 음성 인식 등.                                           |
| **확장성**               | 수평 확장 가능하지만 고차원 데이터나 복잡한 쿼리 처리에는 한계.               | 수백만~수십억 개 벡터 데이터를 효율적으로 처리 가능.                                                  |

**벡터 데이터베이스의 주요 특징**

1. **Approximate Nearest Neighbor (ANN) 검색**  
   - **ANN 알고리즘**을 사용해 유사한 벡터를 빠르게 검색.  
   - 검색 속도가 빠르고, 대규모 데이터셋에서도 효율적으로 동작.

2. **확장성**  
   - 수백만~수십억 개의 벡터 데이터를 처리할 수 있는 구조로 설계.  
   - 대규모 데이터셋에서 고속 검색 및 처리가 가능.

3. **유연성**  
   - 텍스트, 이미지, 오디오 데이터를 임베딩 형태로 변환해 저장 가능.  
   - 다양한 머신러닝 모델과 통합하여 사용자 요구에 맞는 검색 시스템 구축 가능.

**주요 벡터 데이터베이스 비교**

| **이름**      | **특징**                                                                                                                                   | **장점**                                                                                                             | **단점**                                                                                  |
|---------------|-------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------|
| **Chroma**    | 오픈 소스 벡터 데이터베이스, LLM(대규모 언어 모델) 응용에 최적화. Python 노트북 환경에서 간편하게 사용 가능하며 프로덕션으로 확장 가능.                | 간편한 설정, 유사성 검색 및 임베딩 관리 용이, LLM 응용 프로그램에 적합.                                               | 대규모 데이터 처리에서 다른 서비스만큼 최적화되어 있지 않을 수 있음.                                          |
| **Pinecone**  | 완전 관리형 서비스로, 대규모 고차원 데이터의 실시간 처리 및 검색에 최적화.                                                                  | 유지보수 불필요(관리형 서비스), 실시간 대규모 데이터 검색에 강점, 데이터 엔지니어 및 과학자들에게 적합.                 | 오픈 소스가 아니며, 서비스 사용 비용이 발생.                                                              |
| **Weaviate**  | 오픈 소스 기반, OpenAI, Cohere, HuggingFace와의 통합으로 벡터화 작업 용이.                                                                 | 다양한 플랫폼과의 통합 기능, 확장성과 유연성, 고차원 데이터 검색 성능 우수.                                           | 복잡한 설정 및 사용 시 초기 학습 필요.                                                              |
| **Faiss**     | Meta에서 개발한 라이브러리로 대규모 벡터 세트 검색에 최적화. Python 및 GPU 지원으로 성능 극대화.                                              | 고성능 검색(GPU 지원), 대규모 데이터셋 처리 능력, 빠른 속도.                                                          | 데이터베이스가 아닌 라이브러리 형태로 제공되어, 추가적인 환경 설정 및 통합 작업 필요.                                         |
| **Qdrant**    | Rust로 구현된 API 기반 벡터 검색 도구. 빠른 검색과 자원 최적화를 제공하며 정교한 필터링 기능 지원.                                             | 뛰어난 성능(Rust 기반), 정교한 필터링 기능, API 중심의 유연한 설계.                                                    | 커뮤니티와 생태계가 다른 데이터베이스에 비해 상대적으로 작을 수 있음.                                         |

**선택 가이드**
1. **LLM 응용 프로그램**: Chroma, Weaviate.  
2. **완전 관리형 서비스**: Pinecone.  
3. **고성능 및 GPU 지원 필요**: Faiss.  
4. **정교한 필터링과 최적화된 성능**: Qdrant.  

### FAISS

- **공식 문서**: https://faiss.ai/
- **Github**: https://github.com/facebookresearch/faiss

**Faiss(Vector Search Library)**는 Facebook AI Research에서 개발한 **효율적인 벡터 검색 및 밀집 벡터 인덱싱 라이브러리**이다. 대규모 데이터에서 **빠른 유사도 검색과 군집화**를 수행하는 데 최적화되어 있다. 주로 문서 검색, 추천 시스템, 이미지 검색, NLP 모델에서 벡터 임베딩 처리를 지원한다.

**주요 특징**
1. **효율적인 유사도 검색**
   - `k-NN (k-Nearest Neighbors)`를 기반으로 벡터 간 유사도(예: 코사인 유사도, L2 거리)를 계산한다.
   - CPU/GPU 모두 지원하여 대규모 데이터에서도 빠르게 처리 가능하다.

2. **고성능 인덱싱**
   - 다양한 **인덱싱 알고리즘**(Flat, IVF, HNSW, PQ 등)을 지원하여 정확도와 속도 간 균형을 맞출 수 있다.
   - 데이터가 커질수록 효율적으로 검색 성능을 발휘하도록 설계되었다.

3. **확장성**
   - 수억 개의 벡터에서도 성능을 유지하도록 설계되었으며, GPU 병렬 처리를 통해 성능을 극대화한다.

4. **유연성**
   - Python과 C++ API를 제공하며, Scikit-learn이나 PyTorch와 같은 다른 라이브러리와 통합하여 사용 가능하다.

**Faiss의 기본 인덱스 유형**
1. **Flat Index**
   - 모든 벡터를 저장하고 전체 탐색(Brute-Force)을 수행.
   - 정확도가 높지만 대규모 데이터에서는 속도가 느릴 수 있다.

2. **IVF (Inverted File Index)**
   - 벡터를 클러스터링하여 데이터 양을 줄이고 탐색 속도를 높임.
   - 대규모 데이터에서 적합하며, 정확도와 속도 조절 가능.

3. **PQ (Product Quantization)**
   - 벡터를 압축하여 메모리 사용량을 줄이고, 빠른 근사 유사도 검색 수행.

4. **HNSW (Hierarchical Navigable Small World Graphs)**
   - 그래프 기반 알고리즘으로 매우 빠른 근사 유사도 검색 가능.


**Faiss의 주요 사용 사례**
1. **문서 검색**
   - 문서를 벡터로 변환한 후 가장 관련 있는 문서를 검색.
   - NLP 모델의 임베딩과 결합하여 사용.

2. **이미지 검색**
   - 이미지 특징 벡터를 사용하여 비슷한 이미지를 검색.

3. **추천 시스템**
   - 사용자의 행동이나 관심사를 벡터화하여 추천 품목 생성.

4. **클러스터링**
   - 벡터 데이터를 군집화하여 데이터의 구조를 분석.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
import numpy as np

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()
page_contents = [doc.page_content for doc in docs]

embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')
emb_vecs = embeddings.embed_documents(page_contents)    # 페이지별 컨텐츠 임베딩 -> 벡터 리스트

np.array(emb_vecs).shape    # (페이지수, 임베딩 차원수)

In [ ]:
# FAISS 벡터스토어를 이용해 문서들을 임베딩해 로컬에 저장
from langchain_community.vectorstores import FAISS  # FAISS 기반

vector_db = FAISS.from_documents(docs, embeddings)  # docs 를 임베딩해서 FAISS 인덱스 생성
vector_db.save_local('./db/faiss')      # 로컬 경로에 FAISS 인덱스/ 메타데이터 저장


In [ ]:
# 로컬에 저장해놓은 FAISS 벡터스토어 로드
vector_db = FAISS.load_local(
    './db/faiss',   # 경로
    embeddings,     # 로드시 사용할 임베딩 모델
    allow_dangerous_deserialization= True   # 신뢰된 파일만 사용(pickle 역직렬화 허용)
)

In [ ]:
search_result = vector_db.similarity_search(
    query = '학교 선생님이 아끼는 해부학 책은 누가 찢었는가?',  # 쿼리 : 한글
    k= 4    # 상위 4개 Document
)

search_result   # list[Document]

In [ ]:
for i, doc in enumerate(search_result, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f'{i}번째 {doc.metadata['page_label']} page: {doc.page_content}')


### VectorStoreRetriever

리트리버는 벡터DB의 검색 기능을 표준화하고 추상화하여 LangChain 생태계에서 재사용성을 높이는 어댑터(Adapter) 역할을 수행한다.

벡터 저장소를 **`Retriever`라는 표준 인터페이스(Runnable)로 변환**한 뒤 실행하는 방식이다.

단순 유사도 검색뿐만 아니라, `search_type` 설정을 통해 **MMR(다양성 확보), 임계값 필터링(score_threshold)** 등 고급 검색 로직을 쉽게 적용할 수 있다.

**LCEL(LangChain Expression Language)** 파이프라인(`chain = retriever | llm`)에 즉시 통합 가능하다. 코드 수정 없이 검색 알고리즘만 교체하기 쉽다.

In [ ]:
# VectorStore 를 Retriever 인터페이스 변환
retriever = vector_db.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs ={            # 검색 파라미터 묶음
        'k': 3
    }
)

search_result = retriever.invoke('마을 무덤의 남자는 누가 죽였는가?')

for i, doc in enumerate(search_result, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f'{i}번째 {doc.metadata['page_label']} page: {doc.page_content}')


In [ ]:
# 벡터 검색(Retriever) 결과를 Context 에 넣고, PDF 기반 RAG 답변을 생성하는 코드
retriever = vector_db.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs ={            # 검색 파라미터 묶음
        'k': 3
    }
)




prompt = PromptTemplate.from_template('''
        사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.    
        Context : {context}
        Question : {question}
    ''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser()


chain = (
    {'question': RunnablePassthrough(), 'context': retriever | format_docs } | prompt | llm | output_parser
)


# 음식내용 조회 RAG
- 데이터셋 : fine_food_reviews_1k.csv
- 벡터 DB 구성
- Retriever + llm 체인으로 리뷰 조회하는 기능

In [ ]:
import pandas as pd

df = pd.read_csv('fine_food_reviews_1k.csv')
data = df['Text'].to_list()

In [ ]:
vector_store = FAISS.from_texts(data, embeddings)



In [ ]:
# Retriever Chain 요소 구성 (코사인 유사도, 상위 10개 리뷰 검색)
retriever = vector_store.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs ={            # 검색 파라미터 묶음
        'k': 10
    }
)

# prompt Chain 요소 구성 (context, question 입력받음)
prompt = PromptTemplate.from_template('''
        사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.    
        Context : {context}
        Question : {question}
    ''')

# llm/ 문자열 반환 parser 생성
llm = init_chat_model('openai:gpt-5.6-luna')

output_parser = StrOutputParser()


# 체인구성 : question은 입력값 그대로 전달, context 는 검색 + 문서 합치기 | 최종프로젝스 완성 | LLM 호출
chain = (
    {'question': RunnablePassthrough(), 'context': retriever | format_docs } | prompt | llm | output_parser
)

# fresh fruit 관련 답변 출력
print(chain.invoke('fresh fruit'))